# MetroPT-3 data exploration

Run `uv run python -m research download` and `uv run python -m research prepare` from the repository first. Reusable inspection logic lives in `research/eda.py`.

Source: [UCI MetroPT-3](https://archive.ics.uci.edu/dataset/791/metropt+3+dataset), CC BY 4.0. Failure intervals are weak annotations; unreported data are unknown.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from anomaly_detection.config import load_config
from anomaly_detection.data.schema import CONTINUOUS_COLUMNS, DIGITAL_COLUMNS

root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)
config = load_config(root / "configs/experiments.yaml")
observations = pd.read_parquet(root / config["data"]["processed_path"])
observations.head()

## Cadence and gaps
Inspect the measured spacing before building temporal windows. Minute snapshots use past readings only and do not fill missing time.

In [ ]:
quality = json.loads((root / "artifacts/data/data_quality.json").read_text())
quality

In [ ]:
observations[CONTINUOUS_COLUMNS].describe().T

In [ ]:
{column: observations[column].value_counts().to_dict() for column in DIGITAL_COLUMNS}

In [ ]:
pd.read_csv(root / "artifacts/data/failure_timeline.csv")

## Sensors and operating states
Shading marks reported failures. Current bands describe compressor operation, not failure labels.

In [ ]:
for figure in [
    "sensor_timeline.png",
    "sensor_distributions.png",
    "digital_signals.png",
    "sensor_correlations.png",
    "operating_states.png",
]:
    display(Image(filename=str(root / "reports" / "figures" / figure)))

## Splits
February: training and normal development. March–June 14: validation. June 15 onward: test. See `docs/data.md`.